# Machine Learning Industrial Training Project
**Presented by:** Dev Gupta  
**Roll No.:** CS-AI(059)  
**Institution:** Jaipur Engineering College & Research Centre  
**Topic:** Machine Learning

This notebook implements the two practical case studies described in the training presentation:

1. **CreditWise Loan Approval System** — supervised classification.
2. **SmartCart Customer Segmentation** — unsupervised clustering.

It also includes a small Q-learning demonstration to illustrate the Reinforcement Learning concepts from the presentation.

> **Important data note:** The presentation describes a 1,000-row loan dataset and a retail customer dataset, but the actual CSV files are not included in the uploaded PPT. Therefore, this notebook creates reproducible **synthetic/demo data** with similar feature themes. The metrics and clusters produced here are demonstration results, not the original results reported in the presentation.

## 1. Install / import libraries
If a library is missing, uncomment and run the install command.

In [ ]:
# %pip install numpy pandas matplotlib scikit-learn seaborn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully.")

# Project 1: CreditWise Loan Approval System
## 2. Problem statement
SecureTrust Bank wants to reduce slow and inconsistent manual loan verification. We will train a supervised ML classification model to predict whether a loan application is **Approved** or **Rejected**.

The presentation describes 1,000 applicant records and 20 features, including income, credit score, DTI ratio, employment status and loan purpose. Since the original dataset was not supplied, the next cell creates a similar synthetic dataset for practice.

In [ ]:
# Create a reproducible synthetic loan-applicant dataset (1,000 records)
n = 1000

income = np.random.lognormal(mean=10.7, sigma=0.45, size=n)
credit_score = np.clip(np.random.normal(660, 75, n), 300, 850)
loan_amount = np.random.lognormal(mean=11.0, sigma=0.5, size=n)
employment_years = np.clip(np.random.normal(6, 4, n), 0, 35)
existing_debt = np.random.lognormal(mean=9.5, sigma=0.6, size=n)
dti_ratio = np.clip(existing_debt / np.maximum(income, 1) * 12, 0.02, 0.85)

age = np.random.randint(21, 66, n)
employment_status = np.random.choice(["Salaried", "Self-employed", "Unemployed"], n, p=[0.65, 0.25, 0.10])
loan_purpose = np.random.choice(["Home", "Auto", "Education", "Personal", "Business"], n)
education = np.random.choice(["High School", "Graduate", "Postgraduate"], n)
marital_status = np.random.choice(["Single", "Married", "Other"], n)
property_ownership = np.random.choice(["Own", "Rent", "Mortgage"], n)
dependents = np.random.poisson(1.3, n)
bank_accounts = np.random.randint(1, 6, n)
late_payments = np.random.poisson(1.0, n)
loan_term = np.random.choice([12, 24, 36, 48, 60], n)
savings = np.random.lognormal(mean=9.2, sigma=0.7, size=n)
monthly_expenses = np.random.lognormal(mean=8.8, sigma=0.4, size=n)
previous_loans = np.random.randint(0, 5, n)
residence_years = np.random.randint(0, 25, n)
loan_to_income = loan_amount / np.maximum(income, 1)

# Approval probability is generated from plausible risk-related factors.
risk_score = (
    0.012 * (credit_score - 600)
    - 3.0 * dti_ratio
    + 0.000012 * (income - 30000)
    + 0.07 * employment_years
    - 0.35 * late_payments
    + np.where(employment_status == "Unemployed", -1.0, 0)
    + np.random.normal(0, 0.8, n)
)
prob_approved = 1 / (1 + np.exp(-risk_score))
approved = (prob_approved > np.quantile(prob_approved, 0.702)).astype(int)

loan_df = pd.DataFrame({
    "Income": income, "Credit_Score": credit_score, "Loan_Amount": loan_amount,
    "Employment_Years": employment_years, "Existing_Debt": existing_debt,
    "DTI_Ratio": dti_ratio, "Age": age, "Employment_Status": employment_status,
    "Loan_Purpose": loan_purpose, "Education": education,
    "Marital_Status": marital_status, "Property_Ownership": property_ownership,
    "Dependents": dependents, "Bank_Accounts": bank_accounts,
    "Late_Payments": late_payments, "Loan_Term": loan_term,
    "Savings": savings, "Monthly_Expenses": monthly_expenses,
    "Previous_Loans": previous_loans, "Residence_Years": residence_years,
    "Loan_Approved": approved
})

# Add a small amount of missing data to demonstrate imputation.
for col in ["Income", "Credit_Score", "Employment_Status", "Loan_Purpose"]:
    missing_idx = np.random.choice(loan_df.index, size=20, replace=False)
    loan_df.loc[missing_idx, col] = np.nan

print("Dataset shape:", loan_df.shape)
display(loan_df.head())
print("\nTarget distribution:")
print(loan_df["Loan_Approved"].value_counts(normalize=True).rename("proportion"))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
loan_df.describe(include="all").T

In [ ]:
approval_counts = loan_df["Loan_Approved"].map({0: "Rejected", 1: "Approved"}).value_counts()
approval_counts.plot(kind="bar", title="Loan Approval Distribution", xlabel="Decision", ylabel="Number of applicants")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Approval percentage: {loan_df['Loan_Approved'].mean() * 100:.1f}%")

In [ ]:
numeric_for_corr = loan_df.select_dtypes(include=np.number)
plt.figure(figsize=(13, 8))
plt.imshow(numeric_for_corr.corr(), aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns, rotation=90)
plt.yticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns)
plt.title("Numeric Feature Correlation")
plt.tight_layout()
plt.show()

## 4. Data preprocessing and train-test split
- Numeric missing values: median imputation.
- Categorical missing values: most-frequent imputation.
- Categorical columns: One-Hot Encoding.
- Numeric columns: StandardScaler.
- Split: 80% training and 20% testing.

In [ ]:
X = loan_df.drop(columns="Loan_Approved")
y = loan_df["Loan_Approved"]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Train and compare classification models

In [ ]:
models = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7)
}

results = []
trained_models = {}

for name, estimator in models.items():
    # GaussianNB requires dense input after one-hot encoding.
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("to_dense", __import__("sklearn").preprocessing.FunctionTransformer(
            lambda x: x.toarray() if hasattr(x, "toarray") else x
        )),
        ("classifier", estimator)
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(results_df)

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)

print("Best model on this synthetic test split:", best_name)
print(classification_report(y_test, best_predictions, target_names=["Rejected", "Approved"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, display_labels=["Rejected", "Approved"], cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 6. Feature engineering: squared DTI and credit score
The presentation mentions adding `DTI_Ratio²` and `Credit_Score²` to capture non-linear relationships. We add these two columns and compare the same model families again.

In [ ]:
X_fe = X.copy()
X_fe["DTI_Ratio_Squared"] = X_fe["DTI_Ratio"] ** 2
X_fe["Credit_Score_Squared"] = X_fe["Credit_Score"] ** 2

num_fe = X_fe.select_dtypes(include=np.number).columns.tolist()
cat_fe = X_fe.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_fe = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_fe),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_fe)
])

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

fe_results = []
fe_models = {}
for name, estimator in models.items():
    model = Pipeline(steps=[
        ("preprocessor", preprocessor_fe),
        ("to_dense", __import__("sklearn").preprocessing.FunctionTransformer(
            lambda x: x.toarray() if hasattr(x, "toarray") else x
        )),
        ("classifier", estimator)
    ])
    model.fit(X_train_fe, y_train_fe)
    pred = model.predict(X_test_fe)
    fe_models[name] = model
    fe_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_fe, pred),
        "Precision": precision_score(y_test_fe, pred, zero_division=0),
        "Recall": recall_score(y_test_fe, pred, zero_division=0),
        "F1-score": f1_score(y_test_fe, pred, zero_division=0)
    })

fe_results_df = pd.DataFrame(fe_results).sort_values("F1-score", ascending=False)
display(fe_results_df)
print("Note: Results depend on this generated demo dataset and may differ from the presentation.")

# Project 2: SmartCart Customer Segmentation
## 7. Problem statement
A retail smart-cart company wants to understand customer purchasing behaviour and create meaningful customer groups for personalized offers and marketing campaigns.

This is an **unsupervised learning** problem: there is no target label. We will engineer Age, Total_Spending and Total_Children, scale the features, reduce dimensions to three PCA components, and apply K-Means clustering.

In [ ]:
# Generate reproducible synthetic customer data
m = 700
birth_year = np.random.randint(1955, 2005, m)
signup_year = np.random.randint(2015, 2024, m)
income_customer = np.random.normal(65000, 22000, m).clip(15000, 180000)
children_home = np.random.poisson(0.8, m)
teens_home = np.random.poisson(0.5, m)

# Spending on product categories
spend_wines = np.random.gamma(2.0, 45, m)
spend_fruits = np.random.gamma(1.8, 18, m)
spend_meat = np.random.gamma(2.2, 35, m)
spend_fish = np.random.gamma(1.5, 15, m)
spend_sweets = np.random.gamma(1.7, 20, m)
spend_gold = np.random.gamma(1.6, 25, m)

customer_df = pd.DataFrame({
    "Year_Birth": birth_year,
    "Year_Customer": signup_year,
    "Income": income_customer,
    "Kidhome": children_home,
    "Teenhome": teens_home,
    "MntWines": spend_wines,
    "MntFruits": spend_fruits,
    "MntMeatProducts": spend_meat,
    "MntFishProducts": spend_fish,
    "MntSweetProducts": spend_sweets,
    "MntGoldProds": spend_gold,
    "NumWebPurchases": np.random.poisson(4, m),
    "NumStorePurchases": np.random.poisson(5, m),
    "NumCatalogPurchases": np.random.poisson(2, m),
    "Recency": np.random.randint(0, 100, m)
})

# A few missing income values for realistic preprocessing.
customer_df.loc[np.random.choice(customer_df.index, 12, replace=False), "Income"] = np.nan

customer_df["Age"] = 2026 - customer_df["Year_Birth"]
spending_columns = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
customer_df["Total_Spending"] = customer_df[spending_columns].sum(axis=1)
customer_df["Total_Children"] = customer_df["Kidhome"] + customer_df["Teenhome"]

print("Customer dataset shape:", customer_df.shape)
display(customer_df.head())

## 8. Prepare features, scale and apply PCA

In [ ]:
# Remove raw birth year after deriving Age; keep useful customer behaviour features.
cluster_features = [
    "Age", "Income", "Total_Spending", "Total_Children",
    "NumWebPurchases", "NumStorePurchases", "NumCatalogPurchases", "Recency"
]
customer_X = customer_df[cluster_features].copy()

# Median imputation, then standardization.
customer_X = customer_X.fillna(customer_X.median(numeric_only=True))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_X)

# Reduce the standardized data to 3 principal components.
pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("PCA output shape:", X_pca.shape)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("Total variance explained by 3 components:", round(pca.explained_variance_ratio_.sum(), 3))

## 9. K-Means clustering and cluster evaluation

In [ ]:
# Try a few cluster counts and compare silhouette scores.
silhouette_results = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca)
    silhouette_results.append({"k": k, "silhouette_score": silhouette_score(X_pca, labels)})

silhouette_df = pd.DataFrame(silhouette_results)
display(silhouette_df)

best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "k"])
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
customer_df["Cluster"] = kmeans.fit_predict(X_pca)
print("Selected number of clusters:", best_k)
print(customer_df["Cluster"].value_counts().sort_index())

In [ ]:
# 3D PCA cluster visualization
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
    c=customer_df["Cluster"], cmap="viridis", s=25, alpha=0.75
)
ax.set_title("SmartCart Customer Clusters — 3D PCA Projection")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_zlabel("PC 3")
fig.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# Summarize each customer segment to support marketing interpretation.
cluster_summary = customer_df.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    Average_Age=("Age", "mean"),
    Average_Income=("Income", "mean"),
    Average_Total_Spending=("Total_Spending", "mean"),
    Average_Children=("Total_Children", "mean"),
    Average_Web_Purchases=("NumWebPurchases", "mean"),
    Average_Store_Purchases=("NumStorePurchases", "mean")
).round(2)

display(cluster_summary)

## 10. Example business interpretation
Use the cluster summary above to understand how customer groups differ. For example:
- A cluster with higher average spending may respond to loyalty rewards or premium offers.
- A cluster with more web purchases may respond to app notifications and online campaigns.
- A cluster with lower spending may respond to discounts or product bundles.

These are possible marketing interpretations; confirm them with real customer data before making business decisions.

# Project 3: Reinforcement Learning — Q-Learning Demo
The presentation explains the key elements of Reinforcement Learning: **Agent, Environment, State, Action, Reward and Policy**. The following small grid-world example demonstrates how an agent can learn a route to a goal using Q-Learning.

In [ ]:
# Small 4x4 grid world. Start = (0, 0), Goal = (3, 3), one blocked cell.
grid_size = 4
start_state = (0, 0)
goal_state = (3, 3)
blocked = {(1, 1)}
actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right

def step(state, action_index):
    dr, dc = actions[action_index]
    nr = int(np.clip(state[0] + dr, 0, grid_size - 1))
    nc = int(np.clip(state[1] + dc, 0, grid_size - 1))
    next_state = (nr, nc)
    if next_state in blocked:
        next_state = state
    done = next_state == goal_state
    reward = 10 if done else -1
    return next_state, reward, done

q_table = np.zeros((grid_size, grid_size, len(actions)))
alpha, gamma, epsilon = 0.1, 0.95, 0.2

for episode in range(1500):
    state = start_state
    for _ in range(50):
        if np.random.random() < epsilon:
            action = np.random.randint(len(actions))
        else:
            action = int(np.argmax(q_table[state[0], state[1]]))
        next_state, reward, done = step(state, action)
        old_value = q_table[state[0], state[1], action]
        next_max = np.max(q_table[next_state[0], next_state[1]])
        q_table[state[0], state[1], action] = old_value + alpha * (
            reward + gamma * next_max - old_value
        )
        state = next_state
        if done:
            break

# Display the learned best action at each grid location.
action_names = ["↑", "↓", "←", "→"]
policy = np.empty((grid_size, grid_size), dtype=object)
for r in range(grid_size):
    for c in range(grid_size):
        policy[r, c] = "GOAL" if (r, c) == goal_state else (
            "BLOCK" if (r, c) in blocked else action_names[int(np.argmax(q_table[r, c]))]
        )
display(pd.DataFrame(policy, columns=[f"Col {i}" for i in range(grid_size)],
                     index=[f"Row {i}" for i in range(grid_size)]))

# 11. Conclusion
- **CreditWise** demonstrates a supervised classification workflow: data generation, EDA, preprocessing, model training, evaluation and feature engineering.
- **SmartCart** demonstrates unsupervised learning: feature engineering, scaling, PCA, K-Means clustering and customer-segment analysis.
- **Q-Learning** provides a small practical illustration of how an agent learns from rewards while interacting with an environment.

## 12. References
1. Tom M. Mitchell, *Machine Learning*, McGraw Hill Education, 1997.
2. Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, O’Reilly Media.
3. Christopher M. Bishop, *Pattern Recognition and Machine Learning*, Springer.
4. Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, MIT Press.
5. Scikit-learn documentation: https://scikit-learn.org/
6. TensorFlow documentation: https://www.tensorflow.org/

---
**Before submission:** If your trainer/institute has provided the original loan or customer CSV files separately, replace the synthetic-data generation cells with `pd.read_csv("your_file.csv")`, then update the column names to match those files. Run all cells from top to bottom and save the executed notebook.